In [1]:
# =========================================
# [셀 1] 분기 재무 데이터 추출 - 환경 설정
# =========================================
# 목적:
#   - SEC companyfacts에서 분기(Q1~Q4) 재무 데이터 추출
#   - fundamentals_quarterly.parquet 생성
#
# 주의:
#   - companyfacts json.gz 파일이 이미 있어야 함
#   - 01_data_loader.ipynb Cell 5까지 실행된 상태 전제

import os
import gzip
import json
import logging
from pathlib import Path
import pandas as pd
from tqdm import tqdm

# -----------------------------
# 경로 설정 (SSOT)
# -----------------------------
QP2_ROOT = Path(r"C:\QP2")
DATA_DIR = QP2_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
META_DIR = DATA_DIR / "meta"

# -----------------------------
# 로깅
# -----------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("qp2_quarterly")

# -----------------------------
# 유틸
# -----------------------------
def save_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)

def load_json_gz(path: Path) -> dict:
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return json.load(f)

# -----------------------------
# master (ticker 매핑용) 로드
# -----------------------------
MASTER_PATH = META_DIR / "sp500_universe.parquet"
if not MASTER_PATH.exists():
    raise FileNotFoundError(f"master 파일 없음: {MASTER_PATH}")

master = pd.read_parquet(MASTER_PATH)
print(f"✅ [셀1] 환경 설정 완료")
print(f"   master: {len(master)} rows")
print(f"   companyfacts 경로: {RAW_DIR / 'sec' / 'companyfacts'}")

✅ [셀1] 환경 설정 완료
   master: 503 rows
   companyfacts 경로: C:\QP2\data\raw\sec\companyfacts


In [22]:
# =========================================
# [셀 2] companyfacts → fundamentals_quarterly (UNION 방식)
# =========================================
# 목적:
#   - SEC companyfacts에서 분기(Q1~Q4) 재무 데이터 추출
#   - filed_date 포함 (look-ahead 방지용)
#
# 산출물:
#   - data/interim/fundamentals_quarterly.parquet
#
# 핵심 변경:
#   - 태그 UNION 방식: 모든 후보 태그에서 데이터 합침
#   - 연도별로 태그 변경되어도 커버 가능

OUT_FUND_QUARTERLY = INTERIM_DIR / "fundamentals_quarterly.parquet"

# ---- 표준 계정 정의 (태그 전수조사 기반) ----
STD_MAP_Q = {
    "Assets": ["Assets"],
    
    "Liabilities": ["Liabilities"],  # fallback 별도 처리
    
    "StockholdersEquity": [
        "StockholdersEquity",
        "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
    ],
    
    "NetIncomeLoss": [
        "NetIncomeLoss",
        "ProfitLoss",
        "NetIncomeLossAvailableToCommonStockholdersBasic",
        "NetIncomeLossAvailableToCommonStockholdersDiluted",
    ],
    
    "CFO": [
        "NetCashProvidedByUsedInOperatingActivities",
        "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations",
    ],
    
    "CapEx": [
        "PaymentsToAcquirePropertyPlantAndEquipment",
        "CapitalExpenditures",
        "PaymentsToAcquireOtherPropertyPlantAndEquipment",
    ],
    
    "Revenue": [
        "Revenues",
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "RevenueFromContractWithCustomerIncludingAssessedTax",
        "SalesRevenueNet",
        "SalesRevenueGoodsNet",
        "SalesRevenueServicesNet",
        "RevenuesNetOfInterestExpense",
        "RevenuesFromExternalCustomers",
        # 금융주용
        "InterestAndDividendIncomeOperating",
        "InterestIncomeExpenseNet",
    ],
}

LIAB_FALLBACK_SUM_Q = ("LiabilitiesCurrent", "LiabilitiesNoncurrent")

_EMPTY_Q = pd.DataFrame(columns=["fy", "fp", "end", "filed", "unit", "value"])

# -----------------------------
# 함수 정의
# -----------------------------
def _get_quarterly_series(gaap: dict, concept: str) -> pd.DataFrame:
    """분기 데이터만 추출 (Q1, Q2, Q3, Q4)"""
    node = gaap.get(concept)
    if not node:
        return _EMPTY_Q.copy()

    rows = []
    units = node.get("units", {})
    for unit, items in units.items():
        if not isinstance(items, list):
            continue
        for it in items:
            fp = str(it.get("fp", "")).upper()
            if fp not in ["Q1", "Q2", "Q3", "Q4"]:
                continue
            fy = it.get("fy")
            end = it.get("end")
            filed = it.get("filed")
            val = it.get("val")
            if fy is None or end is None or val is None:
                continue
            rows.append({
                "fy": int(fy),
                "fp": fp,
                "end": pd.to_datetime(end, errors="coerce"),
                "filed": pd.to_datetime(filed, errors="coerce"),
                "unit": str(unit),
                "value": pd.to_numeric(val, errors="coerce"),
            })

    df = pd.DataFrame(rows) if rows else _EMPTY_Q.copy()
    df = df.dropna(subset=["fy", "end", "value"])
    if df.empty:
        return _EMPTY_Q.copy()

    df = df.sort_values(["fy", "fp", "filed"]).drop_duplicates(subset=["fy", "fp"], keep="last")
    return df

def _pick_best_unit_q(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return _EMPTY_Q.copy()
    df = df.copy()
    if df["unit"].str.contains("USD", case=False, na=False).any():
        df = df[df["unit"].str.contains("USD", case=False, na=False)].copy()
        df = df.sort_values(["fy", "fp", "filed"]).drop_duplicates(subset=["fy", "fp"], keep="last")
    return df if not df.empty else _EMPTY_Q.copy()

def _series_to_value_q(df: pd.DataFrame, name: str) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame(columns=["fy", "fp", "end", "filed", name])
    out = df[["fy", "fp", "end", "filed", "value"]].rename(columns={"value": name}).copy()
    return out

def _outer_merge_on_fy_fp(base: pd.DataFrame, add: pd.DataFrame) -> pd.DataFrame:
    if base is None or base.empty:
        return add.copy()

    m = base.merge(add, on=["fy", "fp"], how="outer", suffixes=("", "_add"))

    for col in ["end", "filed"]:
        if col in m.columns and f"{col}_add" in m.columns:
            m[col] = pd.to_datetime(m[col], errors="coerce")
            m[f"{col}_add"] = pd.to_datetime(m[f"{col}_add"], errors="coerce")
            m[col] = m[[col, f"{col}_add"]].max(axis=1)
            m = m.drop(columns=[f"{col}_add"])

    return m

def _liabilities_fallback_q(gaap: dict) -> pd.DataFrame:
    """Liabilities가 없을 때 Current + Noncurrent 합산"""
    cur = _pick_best_unit_q(_get_quarterly_series(gaap, LIAB_FALLBACK_SUM_Q[0]))
    non = _pick_best_unit_q(_get_quarterly_series(gaap, LIAB_FALLBACK_SUM_Q[1]))

    if cur.empty and non.empty:
        return _EMPTY_Q.copy()

    cur_v = _series_to_value_q(cur, "cur")
    non_v = _series_to_value_q(non, "non")

    m = cur_v.merge(non_v, on=["fy", "fp"], how="outer", suffixes=("", "_x"))

    for col in ["end", "filed"]:
        if col not in m.columns:
            m[col] = pd.NaT
        if f"{col}_x" in m.columns:
            m[col] = pd.to_datetime(m[col], errors="coerce")
            m[f"{col}_x"] = pd.to_datetime(m[f"{col}_x"], errors="coerce")
            m[col] = m[[col, f"{col}_x"]].max(axis=1)
            m = m.drop(columns=[f"{col}_x"])

    m["cur"] = pd.to_numeric(m.get("cur"), errors="coerce").fillna(0.0)
    m["non"] = pd.to_numeric(m.get("non"), errors="coerce").fillna(0.0)
    m["value"] = m["cur"] + m["non"]
    m["unit"] = "USD"

    out = m[["fy", "fp", "end", "filed", "unit", "value"]].copy()
    out = out.dropna(subset=["fy", "fp"])
    out["fy"] = out["fy"].astype(int)
    out = out.sort_values(["fy", "fp", "filed"]).drop_duplicates(subset=["fy", "fp"], keep="last")
    return out if not out.empty else _EMPTY_Q.copy()

def extract_fundamentals_quarterly(obj: dict, cik: str) -> pd.DataFrame:
    """분기 재무제표 추출 — UNION 방식"""
    gaap = obj.get("facts", {}).get("us-gaap", {})

    series = {}
    for std_name, candidates in STD_MAP_Q.items():
        # ★ UNION 방식: 모든 후보 태그에서 데이터 합치기
        combined = []
        for c in candidates:
            df = _pick_best_unit_q(_get_quarterly_series(gaap, c))
            if not df.empty:
                combined.append(df)
        
        if combined:
            merged = pd.concat(combined, ignore_index=True)
            # (fy, fp) 중복 제거 — filed 최신 우선
            merged = merged.sort_values(["fy", "fp", "filed"]).drop_duplicates(subset=["fy", "fp"], keep="last")
            series[std_name] = merged
        else:
            series[std_name] = _EMPTY_Q.copy()

    # Liabilities fallback (union 결과가 비어있을 때만)
    if series["Liabilities"].empty:
        series["Liabilities"] = _liabilities_fallback_q(gaap)

    # wide 병합
    base = None
    for std_name in ["Assets", "Liabilities", "CFO", "NetIncomeLoss", "Revenue", "StockholdersEquity", "CapEx"]:
        df = series.get(std_name, _EMPTY_Q.copy())
        add = _series_to_value_q(df, std_name)
        base = _outer_merge_on_fy_fp(base, add)

    if base is None or base.empty:
        return pd.DataFrame(columns=["cik", "fy", "fp", "end", "filed", "Assets", "Liabilities", "CFO", "NetIncomeLoss", "Revenue", "StockholdersEquity", "CapEx"])

    base["cik"] = cik

    cols = ["cik", "fy", "fp", "end", "filed", "Assets", "Liabilities", "CFO", "NetIncomeLoss", "Revenue", "StockholdersEquity", "CapEx"]
    for c in cols:
        if c not in base.columns:
            base[c] = pd.NA

    base = base[cols].sort_values(["cik", "fy", "fp"]).reset_index(drop=True)
    return base

# -----------------------------
# 실행
# -----------------------------
paths = sorted((RAW_DIR / "sec" / "companyfacts").glob("*.json.gz"))
if len(paths) == 0:
    raise FileNotFoundError("No companyfacts .json.gz files found.")

logger.info("Extracting fundamentals quarterly (UNION 방식): files=%d", len(paths))

bad_q = []
chunks_q = []
for p in tqdm(paths, total=len(paths)):
    cik = p.stem.replace(".json", "").zfill(10)
    try:
        obj = load_json_gz(p)
        chunks_q.append(extract_fundamentals_quarterly(obj, cik=cik))
    except Exception as e:
        bad_q.append({"file": p.name, "error": str(e)})
        logger.warning("Quarterly extract failed: %s | %s", p.name, e)

fund_q = pd.concat(chunks_q, ignore_index=True) if chunks_q else pd.DataFrame()
if fund_q.empty:
    raise RuntimeError("fundamentals_quarterly is empty.")

# ticker 매핑 추가
fund_q = fund_q.merge(
    master[["cik", "ticker_yahoo", "security_name"]].drop_duplicates(),
    on="cik",
    how="left",
)

save_parquet(fund_q, OUT_FUND_QUARTERLY)

# -----------------------------
# 결과 확인
# -----------------------------
print(f"\n✅ 분기 재무 데이터 저장 완료: {OUT_FUND_QUARTERLY}")
print(f"   rows: {len(fund_q):,}")
print(f"   tickers: {fund_q['ticker_yahoo'].nunique()}")
print(f"   fy 범위: {fund_q['fy'].min()} ~ {fund_q['fy'].max()}")

print("\n=== 컬럼별 NaN 비율 (UNION 방식) ===")
cols = ["Assets", "Liabilities", "CFO", "NetIncomeLoss", "Revenue", "StockholdersEquity", "CapEx"]
for c in cols:
    nan_rate = fund_q[c].isna().mean() * 100
    print(f"  {c:25s}: {nan_rate:5.1f}%")

2026-01-30 15:26:47,058 | INFO | Extracting fundamentals quarterly (UNION 방식): files=500
100%|██████████| 500/500 [07:09<00:00,  1.17it/s]



✅ 분기 재무 데이터 저장 완료: C:\QP2\data\interim\fundamentals_quarterly.parquet
   rows: 22,133
   tickers: 503
   fy 범위: 2009 ~ 2026

=== 컬럼별 NaN 비율 (UNION 방식) ===
  Assets                   :   0.1%
  Liabilities              :   5.1%
  CFO                      :   0.2%
  NetIncomeLoss            :   0.2%
  Revenue                  :   2.8%
  StockholdersEquity       :   0.6%
  CapEx                    :  29.9%


In [26]:
# =========================================
# [셀 3] fundamentals_quarterly 증분 업데이트
# =========================================
# 목적:
#   - 기존 parquet 대비 새로 filed된 데이터만 추가
#   - 전체 재파싱 대신 변경된 파일만 처리
#
# 사용법:
#   - 첫 실행: 셀 2로 전체 생성
#   - 이후: 이 셀로 증분 업데이트
#
# 주의:
#   - companyfacts json.gz 파일이 최신이어야 함
#   - SEC에서 파일 다시 다운로드는 별도 처리 필요

import os
from datetime import datetime

def incremental_update_quarterly():
    """증분 업데이트 실행"""
    
    OUT_PATH = INTERIM_DIR / "fundamentals_quarterly.parquet"
    CHECKPOINT_PATH = INTERIM_DIR / "quarterly_update_checkpoint.json"
    
    # -----------------------------
    # 1) 기존 데이터 + 체크포인트 로드
    # -----------------------------
    if not OUT_PATH.exists():
        print("❌ 기존 parquet 없음. 셀 2를 먼저 실행하세요.")
        return
    
    fund_q_old = pd.read_parquet(OUT_PATH)
    last_filed = fund_q_old["filed"].max()
    
    # 체크포인트: 마지막 업데이트 시 파일 mtime 기록
    file_mtimes = {}
    if CHECKPOINT_PATH.exists():
        import json
        with open(CHECKPOINT_PATH, "r") as f:
            file_mtimes = json.load(f)
    
    print(f"✅ 기존 데이터 로드: {len(fund_q_old):,} rows")
    print(f"   마지막 filed: {last_filed.date() if pd.notna(last_filed) else 'N/A'}")
    
    # -----------------------------
    # 2) 변경된 파일 탐지
    # -----------------------------
    paths = sorted((RAW_DIR / "sec" / "companyfacts").glob("*.json.gz"))
    
    changed_files = []
    for p in paths:
        mtime = os.path.getmtime(p)
        mtime_str = str(mtime)
        
        if p.name not in file_mtimes or file_mtimes[p.name] != mtime_str:
            changed_files.append(p)
    
    if not changed_files:
        print("✅ 변경된 파일 없음. 업데이트 불필요.")
        return
    
    print(f"📁 변경된 파일: {len(changed_files)}개 / 전체 {len(paths)}개")
    
    # -----------------------------
    # 3) 변경된 파일만 재파싱
    # -----------------------------
    chunks_new = []
    updated_ciks = set()
    
    for p in tqdm(changed_files, desc="증분 파싱"):
        cik = p.stem.replace(".json", "").zfill(10)
        updated_ciks.add(cik)
        
        try:
            obj = load_json_gz(p)
            chunks_new.append(extract_fundamentals_quarterly(obj, cik=cik))
        except Exception as e:
            logger.warning(f"증분 파싱 실패: {p.name} | {e}")
    
    if not chunks_new:
        print("⚠️ 새 데이터 없음")
        return
    
    fund_q_new = pd.concat(chunks_new, ignore_index=True)
    
    # ticker 매핑
    fund_q_new = fund_q_new.merge(
        master[["cik", "ticker_yahoo", "security_name"]].drop_duplicates(),
        on="cik",
        how="left",
    )
    
    print(f"   새로 파싱: {len(fund_q_new):,} rows, {len(updated_ciks)} tickers")
    
    # -----------------------------
    # 4) 기존 데이터에서 업데이트된 CIK 제거 + 새 데이터 추가
    # -----------------------------
    fund_q_old_filtered = fund_q_old[~fund_q_old["cik"].isin(updated_ciks)]
    fund_q_merged = pd.concat([fund_q_old_filtered, fund_q_new], ignore_index=True)
    
    # 정렬
    fund_q_merged = fund_q_merged.sort_values(["cik", "fy", "fp"]).reset_index(drop=True)
    
    print(f"   기존 유지: {len(fund_q_old_filtered):,} rows")
    print(f"   최종 병합: {len(fund_q_merged):,} rows")
    
    # -----------------------------
    # 5) 저장 + 체크포인트 갱신
    # -----------------------------
    save_parquet(fund_q_merged, OUT_PATH)
    
    # 체크포인트 갱신 (모든 파일의 mtime 기록)
    new_mtimes = {p.name: str(os.path.getmtime(p)) for p in paths}
    import json
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump(new_mtimes, f)
    
    print(f"\n✅ 증분 업데이트 완료: {OUT_PATH}")
    print(f"   체크포인트 저장: {CHECKPOINT_PATH}")
    
    # NaN 비율 확인
    print("\n=== 컬럼별 NaN 비율 ===")
    cols = ["Assets", "Liabilities", "CFO", "NetIncomeLoss", "Revenue", "StockholdersEquity", "CapEx"]
    for c in cols:
        nan_rate = fund_q_merged[c].isna().mean() * 100
        print(f"  {c:25s}: {nan_rate:5.1f}%")

# -----------------------------
# 실행
# -----------------------------
incremental_update_quarterly()

✅ 기존 데이터 로드: 22,133 rows
   마지막 filed: 2026-01-23
✅ 변경된 파일 없음. 업데이트 불필요.


In [28]:
# =========================================
# [셀 4] Yahoo 주가 증분 업데이트
# =========================================
# 목적:
#   - 기존 Yahoo 가격 데이터의 마지막 날짜 이후만 다운로드
#   - 종목별 parquet 업데이트
#   - wide/long 패널 재생성
#
# 산출물:
#   - data/raw/yahoo/{TICKER}.parquet (증분)
#   - data/interim/yahoo_adjclose_wide.parquet (재생성)
#   - data/interim/yahoo_prices_long.parquet (재생성)
#
# 주의:
#   - yfinance 필요: pip install yfinance

import yfinance as yf
from datetime import datetime, timedelta
import pytz

def update_yahoo_prices():
    """Yahoo 주가 증분 업데이트"""
    
    YAHOO_DIR = RAW_DIR / "yahoo"
    YAHOO_DIR.mkdir(parents=True, exist_ok=True)
    
    OUT_WIDE = INTERIM_DIR / "yahoo_adjclose_wide.parquet"
    OUT_LONG = INTERIM_DIR / "yahoo_prices_long.parquet"
    
    # -----------------------------
    # 1) S&P500 티커 로드
    # -----------------------------
    sp500 = pd.read_parquet(META_DIR / "sp500_universe.parquet")
    
    if "ticker_yahoo" in sp500.columns:
        tickers = sp500["ticker_yahoo"].dropna().unique().tolist()
    else:
        tickers = sp500["ticker"].dropna().unique().tolist()
    
    tickers = sorted([str(t).strip() for t in tickers if t])
    print(f"✅ S&P500 티커: {len(tickers)}개")
    
    # -----------------------------
    # 2) 종목별 증분 업데이트
    # -----------------------------
    NY_TZ = pytz.timezone("America/New_York")
    today = datetime.now(NY_TZ).date()
    
    updated = 0
    failed = []
    skipped = 0
    
    for ticker in tqdm(tickers, desc="Yahoo 증분"):
        ticker_path = YAHOO_DIR / f"{ticker}.parquet"
        
        try:
            # 기존 데이터 확인
            if ticker_path.exists():
                existing = pd.read_parquet(ticker_path)
                if "Date" in existing.columns:
                    existing["Date"] = pd.to_datetime(existing["Date"])
                    last_date = existing["Date"].max().date()
                elif existing.index.name == "Date" or isinstance(existing.index, pd.DatetimeIndex):
                    last_date = existing.index.max().date()
                else:
                    last_date = None
                
                # 이미 최신이면 스킵
                if last_date and last_date >= today - timedelta(days=1):
                    skipped += 1
                    continue
                
                start_date = last_date + timedelta(days=1)
            else:
                existing = None
                start_date = datetime(2000, 1, 1).date()
            
            # 다운로드
            df_new = yf.download(
                ticker,
                start=start_date.isoformat(),
                end=(today + timedelta(days=1)).isoformat(),
                progress=False,
                auto_adjust=False
            )
            
            if df_new.empty:
                if existing is None:
                    failed.append(ticker)
                else:
                    skipped += 1
                continue
            
            # 컬럼 정리 (MultiIndex 처리)
            if isinstance(df_new.columns, pd.MultiIndex):
                df_new.columns = df_new.columns.get_level_values(0)
            
            df_new = df_new.reset_index()
            df_new["Date"] = pd.to_datetime(df_new["Date"]).dt.tz_localize(None)
            
            # 기존 데이터와 병합
            if existing is not None:
                if "Date" not in existing.columns and existing.index.name == "Date":
                    existing = existing.reset_index()
                existing["Date"] = pd.to_datetime(existing["Date"]).dt.tz_localize(None)
                df_merged = pd.concat([existing, df_new], ignore_index=True)
                df_merged = df_merged.drop_duplicates(subset=["Date"], keep="last")
                df_merged = df_merged.sort_values("Date").reset_index(drop=True)
            else:
                df_merged = df_new
            
            # 저장
            df_merged.to_parquet(ticker_path, index=False)
            updated += 1
            
        except Exception as e:
            failed.append(ticker)
    
    print(f"\n✅ Yahoo 증분 완료")
    print(f"   업데이트: {updated}개")
    print(f"   스킵(최신): {skipped}개")
    print(f"   실패: {len(failed)}개 {failed[:10]}{'...' if len(failed) > 10 else ''}")
    
    # -----------------------------
    # 3) Wide/Long 패널 재생성
    # -----------------------------
    print("\n📊 패널 재생성 중...")
    
    all_data = []
    for ticker in tqdm(tickers, desc="패널 생성"):
        ticker_path = YAHOO_DIR / f"{ticker}.parquet"
        if not ticker_path.exists():
            continue
        
        try:
            df = pd.read_parquet(ticker_path)
            if "Date" not in df.columns:
                df = df.reset_index()
            
            df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None)
            
            # Adj Close 컬럼 찾기
            adj_col = None
            for col in ["Adj Close", "Adj_Close", "AdjClose", "adj_close"]:
                if col in df.columns:
                    adj_col = col
                    break
            
            if adj_col is None:
                continue
            
            df_slim = df[["Date", adj_col]].copy()
            df_slim.columns = ["date", "adj_close"]
            df_slim["ticker"] = ticker
            all_data.append(df_slim)
            
        except Exception as e:
            pass
    
    if not all_data:
        print("❌ 데이터 없음")
        return
    
    long_df = pd.concat(all_data, ignore_index=True)
    long_df = long_df.dropna(subset=["adj_close"])
    
    # Wide format
    wide_df = long_df.pivot(index="date", columns="ticker", values="adj_close")
    wide_df = wide_df.sort_index()
    
    # 저장
    long_df.to_parquet(OUT_LONG, index=False)
    wide_df.to_parquet(OUT_WIDE)
    
    print(f"\n✅ 패널 저장 완료")
    print(f"   Long: {OUT_LONG} ({len(long_df):,} rows)")
    print(f"   Wide: {OUT_WIDE} ({wide_df.shape[0]} dates × {wide_df.shape[1]} tickers)")
    print(f"   기간: {wide_df.index.min().date()} ~ {wide_df.index.max().date()}")

# -----------------------------
# 실행
# -----------------------------
update_yahoo_prices()

✅ S&P500 티커: 503개


Yahoo 증분: 100%|██████████| 503/503 [00:04<00:00, 123.70it/s]



✅ Yahoo 증분 완료
   업데이트: 0개
   스킵(최신): 503개
   실패: 0개 []

📊 패널 재생성 중...


패널 생성: 100%|██████████| 503/503 [00:04<00:00, 101.67it/s]



✅ 패널 저장 완료
   Long: C:\QP2\data\interim\yahoo_prices_long.parquet (4,343,487 rows)
   Wide: C:\QP2\data\interim\yahoo_adjclose_wide.parquet (16127 dates × 503 tickers)
   기간: 1962-01-02 ~ 2026-01-29


In [29]:
# =========================================
# [셀 5] 01_DataLoader_Quarterly.ipynb 요약
# =========================================

# =============================================================================
# [노트북 목적]
# =============================================================================
# SEC EDGAR companyfacts에서 분기(Q1~Q4) 재무 데이터 추출
# Yahoo Finance 주가 데이터 증분 업데이트
# 
# =============================================================================
# [산출물]
# =============================================================================
# 1) data/interim/fundamentals_quarterly.parquet
#    - 컬럼: cik, fy, fp, end, filed, Assets, Liabilities, CFO, 
#            NetIncomeLoss, Revenue, StockholdersEquity, CapEx,
#            ticker_yahoo, security_name
#    - rows: 22,127 | tickers: 503 | fy: 2009~2026
#    - filed 포함 → look-ahead bias 방지 가능
#
# 2) data/interim/yahoo_adjclose_wide.parquet
#    - 16,127 dates × 503 tickers
#    - 기간: 1962-01-02 ~ 2026-01-29
#
# 3) data/interim/yahoo_prices_long.parquet
#    - 4,343,487 rows (date, ticker, adj_close)
#
# 4) data/interim/quarterly_update_checkpoint.json
#    - SEC 파일 mtime 기록 (증분 업데이트용)
#
# =============================================================================
# [NaN 비율 (UNION 방식 적용 후)]
# =============================================================================
# Assets              :  0.1%  ✅
# Liabilities         :  5.1%  ✅ (초기 연도 문제)
# CFO                 :  0.2%  ✅
# NetIncomeLoss       :  0.2%  ✅
# Revenue             :  2.8%  ✅
# StockholdersEquity  :  0.6%  ✅
# CapEx               : 29.9%  ⚠️ (보고 안 하는 기업 많음, 불가피)
#
# =============================================================================
# [핵심 해결책: UNION 방식]
# =============================================================================
# 문제: 회사마다/연도마다 다른 태그 사용 (예: AAPL Revenue)
#   - 2009~2018: SalesRevenueNet
#   - 2019~2025: RevenueFromContractWithCustomerExcludingAssessedTax
#
# 해결: 모든 후보 태그에서 데이터 합치고 (fy, fp) 기준 중복 제거
#   - Revenue: 41% → 2.8%
#   - CapEx: 100% → 29.9%
#
# =============================================================================
# [셀 구조]
# =============================================================================
# 셀 1: 환경 설정 (경로, 유틸, master 로드)
# 셀 2: fundamentals_quarterly 전체 파싱 (UNION 방식)
# 셀 3: fundamentals_quarterly 증분 업데이트
# 셀 4: Yahoo 주가 증분 업데이트
# 셀 5: 요약 (현재)
#
# =============================================================================
# [사용법]
# =============================================================================
# 첫 실행: 셀 1 → 셀 2 → 셀 4 (전체 생성)
# 이후:   셀 1 → 셀 3 → 셀 4 (증분 업데이트)
#
# 또는: C:\QP2\UPDATE.bat 실행 (원클릭 업데이트)
#
# =============================================================================
# [다음 작업]
# =============================================================================
# 02_Factor_04_A_integrated.ipynb에서:
# - fundamentals_quarterly.parquet 로드
# - 분기 데이터 기반 촉매 재계산
# - LAG_DAYS 민감도 테스트
#
# =============================================================================

print("✅ [셀 5] 01_DataLoader_Quarterly.ipynb 요약 완료")
print("다음: 02_Factor_04_A_integrated.ipynb에서 분기 데이터 기반 촉매 재계산")

✅ [셀 5] 01_DataLoader_Quarterly.ipynb 요약 완료
다음: 02_Factor_04_A_integrated.ipynb에서 분기 데이터 기반 촉매 재계산
